EMA-Based Sharpe Ratio Approach (Current Implementation)

This is the current working approach that uses Exponential Moving Average (EMA)
for tracking mean and variance, then calculates a Sharpe-like reward.

This approach achieved Test Sharpe: 1.78 (technical) and 1.92 (sentiment)

In [1]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
from pathlib import Path
import json
import math

from stable_baselines3 import PPO, SAC, A2C
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback
import torch as th

# Import the existing environment (which uses EMA Sharpe)
from environments_part1 import TechnicalAgentEnv, SentimentAgentEnv

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
class ValidationCallback(BaseCallback):
    """Validation callback for early stopping."""
    
    def __init__(self, val_env, eval_freq: int = 5000, patience: int = 10,
                 save_path: str = None, verbose: int = 1):
        super().__init__(verbose)
        self.val_env = val_env
        self.eval_freq = eval_freq
        self.patience = patience
        self.save_path = save_path
        self.best_sharpe = -np.inf
        self.no_improve = 0
        
    def _on_step(self):
        if self.eval_freq <= 0 or (self.n_calls % self.eval_freq) != 0:
            return True
        
        val_sharpe = self.val_env.run_full_pass(self.model)
        
        if self.verbose:
            print(f"[Step {self.num_timesteps:,}] Val Sharpe: {val_sharpe:.3f} (Best: {self.best_sharpe:.3f})")
        
        if val_sharpe > self.best_sharpe + 1e-6:
            self.best_sharpe = val_sharpe
            self.no_improve = 0
            
            if self.save_path:
                self.model.save(self.save_path)
                if self.verbose:
                    print(f"  New best model saved")
        else:
            self.no_improve += 1
            if self.no_improve >= self.patience:
                if self.verbose:
                    print(f"  Early stopping")
                return False
        
        return True


def train_ema_sharpe(agent_type: str, algorithm: str, config: dict, verbose: bool = True):
    """Train agent with EMA-based Sharpe reward."""
    
    if verbose:
        print(f"\n{'='*70}")
        print(f"Training {agent_type.upper()} Agent with {algorithm} (EMA Sharpe)")
        print(f"{'='*70}")
    
    # Create environments (use existing implementation)
    if agent_type == 'technical':
        train_env = TechnicalAgentEnv(
            '../data_hierarchical', 'train',
            rolling_vol_window=config['rolling_vol_window'],
            softmax_temperature=config['softmax_temperature'],
            random_start=True
        )
        val_env = TechnicalAgentEnv(
            '../data_hierarchical', 'val',
            rolling_vol_window=config['rolling_vol_window'],
            softmax_temperature=config['softmax_temperature']
        )
        test_env = TechnicalAgentEnv(
            '../data_hierarchical', 'test',
            rolling_vol_window=config['rolling_vol_window'],
            softmax_temperature=config['softmax_temperature']
        )
    else:
        train_env = SentimentAgentEnv(
            '../data_hierarchical', 'train',
            rolling_vol_window=config['rolling_vol_window'],
            softmax_temperature=config['softmax_temperature'],
            random_start=True
        )
        val_env = SentimentAgentEnv(
            '../data_hierarchical', 'val',
            rolling_vol_window=config['rolling_vol_window'],
            softmax_temperature=config['softmax_temperature']
        )
        test_env = SentimentAgentEnv(
            '../data_hierarchical', 'test',
            rolling_vol_window=config['rolling_vol_window'],
            softmax_temperature=config['softmax_temperature']
        )
    
    vec_env = DummyVecEnv([lambda: train_env])
    
    # Create model
    model_kwargs = {
        'policy': 'MlpPolicy',
        'env': vec_env,
        'learning_rate': config['learning_rate'],
        'gamma': config['gamma'],
        'seed': config['seed'],
        'verbose': 0 if not verbose else 1,
    }
    
    if algorithm == 'PPO':
        model = PPO(
            **model_kwargs,
            n_steps=2048,
            batch_size=512,
            policy_kwargs=dict(activation_fn=th.nn.ReLU, net_arch=[256, 256]),
        )
    elif algorithm == 'SAC':
        model = SAC(
            **model_kwargs,
            buffer_size=200_000,
            batch_size=1024,
            policy_kwargs=dict(activation_fn=th.nn.ReLU, net_arch=dict(pi=[256, 256], qf=[256, 256]))
        )
    elif algorithm == 'A2C':
        model = A2C(
            **model_kwargs,
            n_steps=5,
            policy_kwargs=dict(activation_fn=th.nn.ReLU, net_arch=[256, 256])
        )
    else:
        raise ValueError(f"Unknown algorithm: {algorithm}")
    
    # Training
    save_path = Path('../models') / agent_type / 'ema_sharpe' / f"ema_sharpe_{agent_type}_{algorithm.lower()}.zip"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    callback = ValidationCallback(
        val_env=val_env,
        eval_freq=config['eval_freq'],
        patience=config['patience'],
        save_path=str(save_path),
        verbose=1 if verbose else 0
    )
    
    model.learn(total_timesteps=config['total_steps'], callback=callback)
    
    # Load best and evaluate
    if save_path.exists():
        if algorithm == 'PPO':
            model = PPO.load(str(save_path), env=vec_env)
        elif algorithm == 'SAC':
            model = SAC.load(str(save_path), env=vec_env)
        elif algorithm == 'A2C':
            model = A2C.load(str(save_path), env=vec_env)
    
    train_sharpe = train_env.run_full_pass(model)
    val_sharpe = val_env.run_full_pass(model)
    test_sharpe = test_env.run_full_pass(model)
    
    if verbose:
        print(f"\nResults:")
        print(f"  Train Sharpe: {train_sharpe:.3f}")
        print(f"  Val Sharpe:   {val_sharpe:.3f}")
        print(f"  Test Sharpe:  {test_sharpe:.3f}")
    
    return {
        'agent_type': agent_type,
        'algorithm': algorithm,
        'reward_type': 'ema_sharpe',
        'train_sharpe': train_sharpe,
        'val_sharpe': val_sharpe,
        'test_sharpe': test_sharpe,
        'model_path': str(save_path)
    }


In [3]:
print("="*70)
print("EMA-BASED SHARPE RATIO APPROACH (CURRENT)")
print("="*70)

# Configuration
CONFIG = {
    'data_dir': 'data_hierarchical',
    'models_dir': 'models',
    'total_steps': 300_000,
    'eval_freq': 5_000,
    'patience': 5,
    'learning_rate': 3e-4,
    'gamma': 0.99,
    'rolling_vol_window': 12,  # EMA window
    'softmax_temperature': 3.0,
    'seed': 42,
}

Path(CONFIG['models_dir']).mkdir(exist_ok=True)

ALGORITHMS = ['PPO', 'SAC', 'A2C']

# Train Technical Agent
print("\n" + "="*70)
print("TRAINING TECHNICAL AGENT (EMA Sharpe)")
print("="*70)

tech_results = []
for algo in ALGORITHMS:
    result = train_ema_sharpe('technical', algo, CONFIG, verbose=True)
    tech_results.append(result)

tech_df = pd.DataFrame(tech_results).sort_values('val_sharpe', ascending=False)

print("\n" + "="*70)
print("TECHNICAL AGENT RESULTS")
print("="*70)
print(tech_df.to_string(index=False))

best_tech = tech_df.iloc[0].to_dict()
print(f"\nBest: {best_tech['algorithm']} (Test Sharpe: {best_tech['test_sharpe']:.3f})")

# Train Sentiment Agent
print("\n" + "="*70)
print("TRAINING SENTIMENT AGENT (EMA Sharpe)")
print("="*70)

sent_results = []
for algo in ALGORITHMS:
    result = train_ema_sharpe('sentiment', algo, CONFIG, verbose=True)
    sent_results.append(result)

sent_df = pd.DataFrame(sent_results).sort_values('val_sharpe', ascending=False)

print("\n" + "="*70)
print("SENTIMENT AGENT RESULTS")
print("="*70)
print(sent_df.to_string(index=False))

best_sent = sent_df.iloc[0].to_dict()
print(f"\nBest: {best_sent['algorithm']} (Test Sharpe: {best_sent['test_sharpe']:.3f})")

# Save results
results = {
    'approach': 'ema_sharpe',
    'technical': best_tech,
    'sentiment': best_sent,
    'config': CONFIG,
    'all_results': {
        'technical': tech_results,
        'sentiment': sent_results
    }
}

results_path = Path(CONFIG['models_dir']) / 'ema_sharpe_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)

print(f"\nResults saved to {results_path}")

print("\n" + "="*70)
print("EMA SHARPE TRAINING COMPLETE")
print("="*70)
print(f"\nTechnical: {best_tech['algorithm']} - Test Sharpe: {best_tech['test_sharpe']:.3f}")
print(f"Sentiment: {best_sent['algorithm']} - Test Sharpe: {best_sent['test_sharpe']:.3f}")

results


EMA-BASED SHARPE RATIO APPROACH (CURRENT)

TRAINING TECHNICAL AGENT (EMA Sharpe)

Training TECHNICAL Agent with PPO (EMA Sharpe)
TechnicalAgentEnv (train): 201 dates, 7 assets, 20 features
TechnicalAgentEnv (val): 66 dates, 7 assets, 20 features
TechnicalAgentEnv (test): 67 dates, 7 assets, 20 features
Using cpu device
-----------------------------
| time/              |      |
|    fps             | 1685 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 1599        |
|    iterations           | 2           |
|    time_elapsed         | 2           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.009506485 |
|    clip_fraction        | 0.0946      |
|    clip_range           | 0.2         |
|    entropy_loss         | -9.94       |
|    ex

{'approach': 'ema_sharpe',
 'technical': {'agent_type': 'technical',
  'algorithm': 'PPO',
  'reward_type': 'ema_sharpe',
  'train_sharpe': 2.608209342286957,
  'val_sharpe': 2.1093757645033806,
  'test_sharpe': 1.7835528246304748,
  'model_path': '../models/technical/ema_sharpe/ema_sharpe_technical_ppo.zip'},
 'sentiment': {'agent_type': 'sentiment',
  'algorithm': 'SAC',
  'reward_type': 'ema_sharpe',
  'train_sharpe': 5.294478106163799,
  'val_sharpe': 3.132738274114622,
  'test_sharpe': 1.9197566219589604,
  'model_path': '../models/sentiment/ema_sharpe/ema_sharpe_sentiment_sac.zip'},
 'config': {'data_dir': 'data_hierarchical',
  'models_dir': 'models',
  'total_steps': 300000,
  'eval_freq': 5000,
  'patience': 5,
  'learning_rate': 0.0003,
  'gamma': 0.99,
  'rolling_vol_window': 12,
  'softmax_temperature': 3.0,
  'seed': 42},
 'all_results': {'technical': [{'agent_type': 'technical',
    'algorithm': 'PPO',
    'reward_type': 'ema_sharpe',
    'train_sharpe': 2.608209342286957